# Modificar un usuario

Cambia el nombre, el email o los dos. Lo que no se envía conserva su valor.

**Cuidado:** este cuaderno cambia de verdad el usuario de tu base.

## La petición

`PATCH /users/:id`, con este cuerpo:

```json
{ "name": "Ada Lovelace" }
```

Cambia `id` por el de un usuario que exista. El cuaderno de crear te da uno, y el de listar te los enseña todos.

## Qué pasa en la base

Cambia esa fila y actualiza su fecha `updatedAt`.

## Las respuestas

| Estado | Qué se ve |
| --- | --- |
| 200 | El usuario con los datos nuevos |
| 400 | `Name is empty` o `Email is invalid` |
| 404 | `User not found`. No hay ningún usuario con ese id |
| 409 | `Email is already used`. Ese email es de otro usuario |
| 500 | `Could not update the user`. No se ha llegado a la base |

## Qué tiene que estar en marcha

Antes de ejecutarlo, lanza `pnpm start` y `pnpm node-http`, cada uno en su terminal.


In [ ]:
# Lanza programas y lee las variables del sistema.
import os
import subprocess
# Permite subir de carpeta hasta encontrar la app.
from pathlib import Path

# Busca la carpeta de la app, para que Node encuentre sus paquetes.
app = Path.cwd()
while app != app.parent and not (app / "src" / "logic").is_dir():
    app = app.parent
if not (app / "src" / "logic").is_dir():
    raise SystemExit("Cannot find the app.")

# El programa de Node que llama a la API.
code = r"""
// Pinta tablas en la salida del cuaderno.
import Table from "cli-table3";

// Dirección de la API.
const ORIGIN = "http://127.0.0.1:3004";

// El id del usuario. Cámbialo por uno que exista.
const id = 1;
// Los cambios. Puedes añadir también el email.
const changes = { name: "Ada Lovelace" };

// Lanza la petición.
const response = await fetch(`${ORIGIN}/users/${id}`, {
  method: "PATCH",
  headers: { "content-type": "application/json" },
  body: JSON.stringify(changes),
});

// Muestra el método, la ruta y el estado de la respuesta.
console.log(`PATCH /users/${id}: ${response.status}`);

// Lee la respuesta. Algunas respuestas no traen nada.
const text = await response.text();
const result = text === "" ? null : JSON.parse(text);

// Si ha fallado, se ve el mensaje y no la tabla.
if (!response.ok) {
  console.log(result?.message ?? "");
  process.exit(0);
}

// Prepara la tabla con los datos de cada usuario.
const table = new Table({
  head: ["id", "name", "email", "createdAt", "updatedAt"],
  style: { head: ["cyan", "bold"] },
});
const user = result;
table.push([user.id, user.name, user.email, user.createdAt, user.updatedAt]);
console.log(table.toString());
"""

# Pide a Node que escriba con colores.
env = os.environ.copy()
env["FORCE_COLOR"] = "1"
env.pop("NO_COLOR", None)

# Lanza Node y muestra lo que escribe.
completed = subprocess.run(
    ["node", "--input-type=module", "-e", code],
    cwd=app,
    capture_output=True,
    text=True,
    env=env,
)
print(completed.stdout, end="")
if completed.returncode != 0:
    print(completed.stderr, end="")
